In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

In [ ]:
# HyperParameters
BATCH_SIZE = 128
LEARNING_RATE = 0.01
EPOCHS = 30
NUM_CLASSES = 10

# GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(torch.__version__)
print(device)

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
class ConvBlock(nn.Sequential):
    def __init__(self, in_planes, out_planes, kernel_size=3, stride=1, groups=1):
        padding = (kernel_size - 1) // 2
        super(ConvBlock, self).__init__(
            nn.Conv2d(in_planes, out_planes, kernel_size, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_planes),
            nn.ReLU6(inplace=True) 
        )

# Inverted Residual Block
class InvertedResidual(nn.Module):
    def __init__(self, input_dim, output_dim, stride, expand_ratio):
        super(InvertedResidual, self).__init__()
        self.stride = stride

        # Expantion Ratio
        hidden_dim = int(round(input_dim * expand_ratio))

        # Skip Connection
        self.use_shortcut = self.stride == 1 and input_dim == output_dim

        layers = []
        if expand_ratio != 1:
            # Expand(Expressiveness)
            layers.append(ConvBlock(input_dim, hidden_dim, kernel_size=1))

        layers.append(ConvBlock(hidden_dim, hidden_dim, stride=stride, groups=hidden_dim))
        layers.append(nn.Conv2d(hidden_dim, output_dim, 1, 1, 0, bias=False))
        layers.append(nn.BatchNorm2d(output_dim))
        
        # Linear Bottleneck -> not use ReLU 
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_shortcut:
            return x + self.conv(x) # x + F(x) 잔차연결
        else:
            return self.conv(x)

In [ ]:
class MobileNetV2(nn.Module):
    def __init__(self, num_classes=10):
        super(MobileNetV2, self).__init__()
        input_channel = 32
        last_channel = 1280
        
        inverted_residual_setting = [
            # t(확장), c(출력채널), n(반복), s(스트라이드)
            [1, 16, 1, 1],
            [6, 24, 2, 1], # stride 2 -> 1로 변경(CIFAR-10)
            [6, 32, 3, 2],
            [6, 64, 4, 2],
            [6, 96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]

        # 첫 번째 Layer에서 Stride 2 -> 1로 변경
        self.features = []
        self.features.append(ConvBlock(3, input_channel, stride=1))

        # Bottleneck Blocks
        for t, c, n, s in inverted_residual_setting:
            output_channel = c
            # 같은 블록을 반복할 때 첫 블록에서만 크기를 줄임.
            for i in range(n):
                stride = s if i == 0 else 1
                self.features.append(InvertedResidual(input_channel, output_channel, stride, expand_ratio=t))
                input_channel = output_channel

        self.features.append(ConvBlock(input_channel, last_channel, kernel_size=1))
        self.features = nn.Sequential(*self.features)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(last_channel, num_classes),
        )

    def forward(self, x):
        x = self.features(x)   # (Batch, 1280, 4, 4)
        x = self.avg_pool(x)   # (Batch, 1280, 1, 1) 
        x = torch.flatten(x, 1)# (Batch, 1280)      
        x = self.classifier(x)
        return x

In [ ]:
model = MobileNetV2(num_classes=NUM_CLASSES)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
from google.colab import drive

# Google Drive 마운트 및 저장 경로 설정
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/{원하는경로}'
os.makedirs(save_dir, exist_ok=True)

print(f"MobileNet_V2 학습 시작")
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()

    # Training
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    scheduler.step()

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # 결과 계산
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    epoch_time = time.time() - start_time

    # Best Model 저장 로직
    if epoch_acc > best_acc:
        best_acc = epoch_acc

        # Best Model 저장
        save_path = os.path.join(save_dir, f"MobileNetV2_best.pth")
        torch.save(model.state_dict(), save_path)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {epoch_loss:.4f} | "
          f"Test Acc: {epoch_acc:.2f}% | "
          f"Time: {epoch_time:.1f}s")

print("MobileNet_V2 학습 종료")
print(f"최종 Best Accuracy: {best_acc:.2f}%")


학습 시작 (Total Epochs: 30)


Epoch 1: 100%|██████████| 391/391 [00:46<00:00,  8.45it/s, acc=35.7, loss=1.71]


Test | Loss: 1.3725 | Acc: 50.07%


Epoch 2: 100%|██████████| 391/391 [00:46<00:00,  8.44it/s, acc=55.1, loss=1.25]


Test | Loss: 1.0920 | Acc: 61.82%


...

Test | Loss: 0.3543 | Acc: 88.98%


Epoch 29: 100%|██████████| 391/391 [00:47<00:00,  8.23it/s, acc=94.7, loss=0.153]


Test | Loss: 0.3564 | Acc: 88.93%


Epoch 30: 100%|██████████| 391/391 [00:47<00:00,  8.23it/s, acc=94.8, loss=0.155]


Test | Loss: 0.3539 | Acc: 89.06%
